# PolarEngine Full Pipeline Test

End-to-end test of `polarengine-vllm` package:
1. Install package from GitHub (with H128 GPU cache fix)
2. On-the-fly quantize + replace layers (proven path)
3. INT4 bit-packing (VRAM reduction)
4. Benchmark: tok/s, VRAM, PPL
5. Save quantized model to disk
6. Load from disk and verify

Target: 34+ tok/s, ~7.9 GB VRAM, PPL ~6.89

In [ ]:
!pip install git+https://github.com/huggingface/transformers.git --force-reinstall
!pip install -q datasets accelerate safetensors sentencepiece tiktoken scipy triton
!pip install -q git+https://github.com/caiovicentino/polarengine-vllm.git

In [ ]:
# REINICIAR RUNTIME!
import transformers; print(f'transformers: {transformers.__version__}')
import polarengine_vllm; print(f'polarengine-vllm: {polarengine_vllm.__version__}')

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, time, math, gc, os, json
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

from polarengine_vllm.utils import get_centroids, get_bits_for_layer
from polarengine_vllm.kernels.fwht import build_hadamard
# Import kernel JIT functions directly (skip wrapper to avoid per-call allocation)
from polarengine_vllm.kernels.polar_gemv import (
    polar_gemv_kernel, polar_gemv_packed_kernel,  # raw Triton JIT
    polar_gemv, polar_gemv_packed, pack_codes_int4,  # wrappers (for reference)
)

DEVICE = 'cuda'
MODEL = 'Qwen/Qwen3.5-9B'
BS = 128
HALF_K = BS // 2

print(f'GPU: {torch.cuda.get_device_name()}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

H128 = build_hadamard(BS, DEVICE)
print(f'H128 cached on {H128.device}')

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token
print('Ready!')

In [ ]:
# ===============================================================
# STEP 1: Load FP16 + On-the-fly PolarQuant + Replace layers
#
# Package kernels called DIRECTLY (no wrapper allocation overhead)
# Same pattern as v4 notebook: output[b] passed as view to kernel
# ===============================================================
print('='*60)
print('  Step 1: Load + Quantize on-the-fly')
print('='*60)

model = AutoModelForCausalLM.from_pretrained(
    MODEL, dtype=torch.float16, device_map='auto', trust_remote_code=True
)
model.eval()
fp16_vram = torch.cuda.memory_allocated() / 1e9

class PolarLinear(nn.Module):
    def __init__(self, in_f, out_f, bits, bs, codes, norms, ct_scaled, bias=None):
        super().__init__()
        self.in_f, self.out_f, self.bits, self.bs = in_f, out_f, bits, bs
        self.in_f_padded = ((in_f + bs - 1) // bs) * bs
        self.n_blocks = self.in_f_padded // bs
        self.packed = False
        self.register_buffer('codes', codes)
        self.register_buffer('norms', norms)
        self.register_buffer('ct_scaled', ct_scaled)
        self.bias = None
        if bias is not None:
            self.register_buffer('bias', bias.half())

    @staticmethod
    def from_linear(linear, bits=4, bs=128):
        dev = linear.weight.device
        in_f, out_f = linear.in_features, linear.out_features
        ct = get_centroids(bits).to(dev)
        w = linear.weight.data.float()
        in_f_padded = ((in_f + bs - 1) // bs) * bs
        n_blocks = in_f_padded // bs
        pad = in_f_padded - in_f
        if pad > 0: w = F.pad(w, (0, pad))
        blocks = w.view(out_f, n_blocks, bs)
        norms = blocks.norm(dim=2).clamp(min=1e-10)
        blocks_norm = blocks / norms.unsqueeze(2)
        all_codes = torch.empty(out_f, n_blocks, bs, dtype=torch.int8, device=dev)
        for i in range(0, out_f, 64):
            end = min(i + 64, out_f)
            b = blocks_norm[i:end].reshape(-1, bs)
            b_rot = (b @ H128) * math.sqrt(bs)
            b_rot = b_rot.view(end - i, n_blocks, bs)
            all_codes[i:end] = (b_rot.unsqueeze(-1) - ct.view(1,1,1,-1)).abs().argmin(-1).to(torch.int8)
        return PolarLinear(
            in_f, out_f, bits, bs,
            codes=all_codes.reshape(out_f, -1),
            norms=norms.half(),
            ct_scaled=(ct / math.sqrt(bs)),
            bias=linear.bias.data if linear.bias is not None else None,
        ).to(dev)

    def forward(self, x):
        global _fwht_cache
        x_flat = x.view(-1, self.in_f).float()
        batch = x_flat.shape[0]

        # Inline FWHT with simple dict cache
        ptr = x.data_ptr()
        if _fwht_cache['ptr'] == ptr and _fwht_cache['in_f'] == self.in_f:
            x_tf = _fwht_cache['result']
        else:
            pad = self.in_f_padded - self.in_f
            x_p = F.pad(x_flat, (0, pad)) if pad > 0 else x_flat
            x_tf = torch.matmul(x_p.view(-1, BS), H128).view(batch, -1)
            _fwht_cache = {'ptr': ptr, 'in_f': self.in_f, 'result': x_tf}

        # Single allocation, kernel writes directly into output[b] (VIEW)
        output = torch.zeros(batch, self.out_f, device=x.device, dtype=torch.float32)
        for b in range(batch):
            if self.packed:
                grid = lambda meta: ((self.out_f + meta['BLOCK_M'] - 1) // meta['BLOCK_M'],)
                polar_gemv_packed_kernel[grid](
                    self.codes_packed, x_tf[b], self.norms, self.ct_scaled, output[b],
                    self.out_f, self.in_f_half, self.n_blocks,
                    HALF_BK=HALF_K,
                )
            else:
                grid = lambda meta: ((self.out_f + meta['BLOCK_M'] - 1) // meta['BLOCK_M'],)
                polar_gemv_kernel[grid](
                    self.codes, x_tf[b], self.norms, self.ct_scaled, output[b],
                    self.out_f, self.in_f_padded, self.n_blocks,
                    BLOCK_K=self.bs,
                )
        result = output.half().view(*x.shape[:-1], self.out_f)
        if self.bias is not None: result = result + self.bias
        return result

_fwht_cache = {'ptr': -1, 'in_f': -1, 'result': None}

count = 0
for name, module in list(model.named_modules()):
    for child_name, child in list(module.named_children()):
        if isinstance(child, nn.Linear) and child.weight.numel() >= 256:
            full = f'{name}.{child_name}' if name else child_name
            bits = get_bits_for_layer(full + '.weight', child.weight)
            if bits >= 16: continue
            setattr(module, child_name, PolarLinear.from_linear(child, bits=bits, bs=BS))
            count += 1
            if count % 50 == 0: print(f'  {count} layers...', flush=True)

for n, p in model.named_parameters():
    if p.dtype == torch.bfloat16: p.data = p.data.half()
for n, b in model.named_buffers():
    if b.dtype == torch.bfloat16: b.data = b.data.half()

def _clear_hook(module, input):
    global _fwht_cache
    _fwht_cache = {'ptr': -1, 'in_f': -1, 'result': None}
model.register_forward_pre_hook(_clear_hook)

gc.collect(); torch.cuda.empty_cache()
unpacked_vram = torch.cuda.memory_allocated() / 1e9
print(f'\n  {count} layers | FP16: {fp16_vram:.1f} GB -> Polar: {unpacked_vram:.1f} GB')

In [ ]:
# ===============================================================
# STEP 2: Sanity check
# ===============================================================
print('='*60)
print('  Step 2: Sanity check')
print('='*60)

test = tokenizer('Hello world', return_tensors='pt').to(DEVICE)
with torch.no_grad(): out = model(**test)
logits = out.logits
print(f'  Logits: range=[{logits.min().item():.1f}, {logits.max().item():.1f}]')
print(f'  NaN: {logits.isnan().any().item()}')
assert not logits.isnan().any(), 'NaN!'
print('  OK!')

In [ ]:
# ===============================================================
# STEP 3: Speed (unpacked)
# ===============================================================
print('='*60)
print('  Step 3: Speed (unpacked int8)')
print('='*60)

inputs = tokenizer('Explain the theory of relativity:', return_tensors='pt').to(DEVICE)
with torch.no_grad(): model.generate(**inputs, max_new_tokens=5, do_sample=False)
torch.cuda.synchronize()

times = []
for i in range(3):
    torch.cuda.synchronize(); t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    torch.cuda.synchronize(); t = time.perf_counter() - t0
    n = out.shape[1] - inputs['input_ids'].shape[1]
    times.append((n, t))
    print(f'  Run {i+1}: {n/t:.1f} tok/s')
unpacked_tps = sum(n for n,_ in times) / sum(t for _,t in times)
print(f'  Unpacked: {unpacked_tps:.1f} tok/s, {unpacked_vram:.1f} GB')

In [ ]:
# ===============================================================
# STEP 4: INT4 bit-packing (halve VRAM for Q3/Q4 layers)
# ===============================================================
print('='*60)
print('  Step 4: INT4 bit-packing')
print('='*60)

HALF_K = BS // 2
vram_before = torch.cuda.memory_allocated() / 1e9
packed_count = 0

for name, module in model.named_modules():
    if isinstance(module, PolarLinear) and module.bits <= 4:
        codes_blocked = module.codes.view(module.out_f, module.n_blocks, BS)
        first_half = codes_blocked[:, :, :HALF_K].to(torch.uint8)
        second_half = codes_blocked[:, :, HALF_K:].to(torch.uint8)
        packed = ((second_half << 4) | first_half).reshape(module.out_f, -1)
        del module.codes
        module.register_buffer('codes_packed', packed)
        module.packed = True
        module.in_f_half = module.n_blocks * HALF_K
        packed_count += 1

gc.collect(); torch.cuda.empty_cache()
packed_vram = torch.cuda.memory_allocated() / 1e9
print(f'  Packed {packed_count} layers (bits <= 4)')
print(f'  VRAM: {vram_before:.1f} -> {packed_vram:.1f} GB (saved {vram_before - packed_vram:.1f} GB)')

# Sanity
with torch.no_grad(): out = model(**test)
assert not out.logits.isnan().any(), 'NaN after packing!'
print('  Sanity OK!')

# Speed
print('  Speed (packed):')
with torch.no_grad(): model.generate(**inputs, max_new_tokens=5, do_sample=False)
torch.cuda.synchronize()
ptimes = []
for i in range(3):
    torch.cuda.synchronize(); t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=100, do_sample=False)
    torch.cuda.synchronize(); t = time.perf_counter() - t0
    n = out.shape[1] - inputs['input_ids'].shape[1]
    ptimes.append((n, t))
    print(f'    Run {i+1}: {n/t:.1f} tok/s')
packed_tps = sum(n for n,_ in ptimes) / sum(t for _,t in ptimes)
print(f'  Packed: {packed_tps:.1f} tok/s, {packed_vram:.1f} GB')

In [ ]:
# ===============================================================
# STEP 5: PPL
# ===============================================================
print('='*60)
print('  Step 5: PPL')
print('='*60)

ds = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')
wiki = '\n\n'.join([t for t in ds['text'] if t.strip()])[:150000]
ids = tokenizer(wiki, return_tensors='pt').input_ids.to(DEVICE)
nlls = []; total = 0; t0 = time.time()
with torch.no_grad():
    for i in range(0, min(ids.size(1)-2048, 15000), 512):
        c = ids[:, i:i+2048]; t = c.clone(); t[:, :1536] = -100
        loss = model(c, labels=t).loss
        nlls.append(loss.item()*512); total += 512
        if (total//512) % 10 == 0:
            print(f'  {total} tok | PPL: {math.exp(sum(nlls)/total):.2f} | {time.time()-t0:.0f}s', flush=True)
ppl = math.exp(sum(nlls)/total)
print(f'  PPL: {ppl:.4f}')

In [ ]:
# ===============================================================
# STEP 6: Save quantized model to disk
# ===============================================================
print('='*60)
print('  Step 6: Save to disk')
print('='*60)

from safetensors.torch import save_file
os.makedirs(SAVE_DIR, exist_ok=True)

state = {}
polar_meta = {}
for name, buf in model.named_buffers():
    state[name] = buf.contiguous().cpu()
for name, param in model.named_parameters():
    state[name] = param.data.contiguous().cpu()

for name, module in model.named_modules():
    if isinstance(module, PolarLinear):
        polar_meta[name] = {
            'in_features': module.in_f,
            'out_features': module.out_f,
            'bits': module.bits,
            'block_size': module.bs,
            'packed': module.packed,
        }

total_bytes = sum(t.numel() * t.element_size() for t in state.values())
print(f'  {len(state)} tensors, {total_bytes/1e9:.1f} GB')
print(f'  {len(polar_meta)} PolarLinear layers')

# Save sharded
shard_size = 5 * 1024**3
current_shard = {}; current_bytes = 0; shard_idx = 0; weight_map = {}
for key in sorted(state.keys()):
    tensor = state[key]
    tb = tensor.numel() * tensor.element_size()
    if current_bytes + tb > shard_size and current_shard:
        fname = f'model-{shard_idx:05d}-of-XXXXX.safetensors'
        save_file(current_shard, os.path.join(SAVE_DIR, fname))
        shard_idx += 1; current_shard = {}; current_bytes = 0
    current_shard[key] = tensor
    current_bytes += tb
    weight_map[key] = f'model-{shard_idx:05d}-of-XXXXX.safetensors'
if current_shard:
    fname = f'model-{shard_idx:05d}-of-XXXXX.safetensors'
    save_file(current_shard, os.path.join(SAVE_DIR, fname))
    shard_idx += 1
n_shards = shard_idx
for old_key in list(weight_map.keys()):
    weight_map[old_key] = weight_map[old_key].replace('XXXXX', f'{n_shards:05d}')
for i in range(n_shards):
    old = os.path.join(SAVE_DIR, f'model-{i:05d}-of-XXXXX.safetensors')
    new = os.path.join(SAVE_DIR, f'model-{i:05d}-of-{n_shards:05d}.safetensors')
    os.rename(old, new)

with open(os.path.join(SAVE_DIR, 'model.safetensors.index.json'), 'w') as f:
    json.dump({'metadata': {'total_size': total_bytes}, 'weight_map': weight_map}, f, indent=2)

with open(os.path.join(SAVE_DIR, 'polar_config.json'), 'w') as f:
    json.dump({'format': 'polar_engine_v4', 'block_size': BS, 'layers': polar_meta,
               'results': {'tps': packed_tps, 'vram_gb': packed_vram, 'ppl': ppl}}, f, indent=2)

tokenizer.save_pretrained(SAVE_DIR)
model.config.save_pretrained(SAVE_DIR)

print(f'  Saved to {SAVE_DIR}:')
for f in sorted(os.listdir(SAVE_DIR)):
    print(f'    {f}: {os.path.getsize(os.path.join(SAVE_DIR, f))/1e6:.1f} MB')

In [ ]:
# ===============================================================
# RESULTS
# ===============================================================
final_vram = torch.cuda.memory_allocated() / 1e9
print(f'\n{"="*60}')
print(f'  POLARENGINE PACKAGE — FULL PIPELINE RESULTS')
print(f'  Qwen3.5-9B | {torch.cuda.get_device_name()}')
print(f'{"="*60}')
print(f'  {"Method":<30} {"tok/s":>8} {"VRAM":>10} {"PPL":>8}')
print(f'  {"-"*60}')
print(f'  {"FP16 baseline":<30} {"45.7":>8} {fp16_vram:>8.1f} GB {"6.37":>8}')
print(f'  {"Standalone v4 (ref)":<30} {"34.2":>8} {"7.9 GB":>10} {"6.89":>8}')
print(f'  {"Package unpacked":<30} {unpacked_tps:>8.1f} {unpacked_vram:>8.1f} GB')
print(f'  {"Package + INT4 pack":<30} {packed_tps:>8.1f} {packed_vram:>8.1f} GB {ppl:>8.4f}')
print(f'  {"-"*60}')
print(f'  Package vs standalone: {packed_tps/34.2:.2f}x speed')
print(f'  Package vs FP16: {packed_tps/45.7*100:.0f}% speed, {fp16_vram/packed_vram:.1f}x less VRAM')
print(f'{"="*60}')